# Задание 3. Сравнение алгоритмов классификации

Цель: Исследовать и сравнить эффективность различных алгоритмов машинного обучения для задачи классификации: KNN, SVM, DT, LR и RF.

In [ ]:
# 1. Импорт библиотек и загрузка данных
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
import joblib
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
iris = load_iris()
X, y = iris.data, iris.target

print(f"Размер датасета: {X.shape}")
print(f"Классы: {iris.target_names}")
print(f"Признаки: {iris.feature_names}")

In [ ]:
# Разделение на обучающую и тестовую выборки (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

## 2. Создание конвейеров для каждого алгоритма

In [ ]:
# Создание конвейеров
knn_pipeline = Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier())])
svm_pipeline = Pipeline([('scaler', StandardScaler()), ('svm', SVC(probability=True))])
dt_pipeline = Pipeline([('scaler', StandardScaler()), ('dt', DecisionTreeClassifier(random_state=42))])
lr_pipeline = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(max_iter=1000, random_state=42))])
rf_pipeline = Pipeline([('scaler', StandardScaler()), ('rf', RandomForestClassifier(random_state=42))])

print("Конвейеры созданы для всех алгоритмов")

## 3. Настройка гиперпараметров

In [ ]:
# Параметры для GridSearchCV
knn_params = {
    'knn__n_neighbors': [3, 5, 7, 9],
    'knn__weights': ['uniform', 'distance']
}

svm_params = {
    'svm__C': [0.1, 1, 10],
    'svm__kernel': ['rbf', 'linear']
}

dt_params = {
    'dt__max_depth': [None, 3, 5, 7],
    'dt__min_samples_split': [2, 5, 10]
}

lr_params = {
    'lr__C': [0.1, 1, 10],
    'lr__solver': ['lbfgs', 'liblinear']
}

rf_params = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [None, 5, 10]
}

print("Параметры для подбора определены")

## 4. Обучение и оценка моделей

In [ ]:
# Обучение KNN
print("Обучение KNN...")
knn_grid = GridSearchCV(knn_pipeline, knn_params, cv=5, scoring='accuracy', n_jobs=-1)
knn_grid.fit(X_train, y_train)
knn_pred = knn_grid.predict(X_test)
knn_accuracy = accuracy_score(y_test, knn_pred)

print(f"KNN - Лучшие параметры: {knn_grid.best_params_}")
print(f"KNN - Accuracy на тесте: {knn_accuracy:.4f}\n")

In [ ]:
# Обучение SVM
print("Обучение SVM...")
svm_grid = GridSearchCV(svm_pipeline, svm_params, cv=5, scoring='accuracy', n_jobs=-1)
svm_grid.fit(X_train, y_train)
svm_pred = svm_grid.predict(X_test)
svm_accuracy = accuracy_score(y_test, svm_pred)

print(f"SVM - Лучшие параметры: {svm_grid.best_params_}")
print(f"SVM - Accuracy на тесте: {svm_accuracy:.4f}\n")

In [ ]:
# Обучение Decision Tree
print("Обучение Decision Tree...")
dt_grid = GridSearchCV(dt_pipeline, dt_params, cv=5, scoring='accuracy', n_jobs=-1)
dt_grid.fit(X_train, y_train)
dt_pred = dt_grid.predict(X_test)
dt_accuracy = accuracy_score(y_test, dt_pred)

print(f"DT - Лучшие параметры: {dt_grid.best_params_}")
print(f"DT - Accuracy на тесте: {dt_accuracy:.4f}\n")

In [ ]:
# Обучение Logistic Regression
print("Обучение Logistic Regression...")
lr_grid = GridSearchCV(lr_pipeline, lr_params, cv=5, scoring='accuracy', n_jobs=-1)
lr_grid.fit(X_train, y_train)
lr_pred = lr_grid.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_pred)

print(f"LR - Лучшие параметры: {lr_grid.best_params_}")
print(f"LR - Accuracy на тесте: {lr_accuracy:.4f}\n")

In [ ]:
# Обучение Random Forest
print("Обучение Random Forest...")
rf_grid = GridSearchCV(rf_pipeline, rf_params, cv=5, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train, y_train)
rf_pred = rf_grid.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)

print(f"RF - Лучшие параметры: {rf_grid.best_params_}")
print(f"RF - Accuracy на тесте: {rf_accuracy:.4f}\n")

In [ ]:
# Сравнение результатов
results = pd.DataFrame({
    'Алгоритм': ['KNN', 'SVM', 'Decision Tree', 'Logistic Regression', 'Random Forest'],
    'Accuracy на тесте': [knn_accuracy, svm_accuracy, dt_accuracy, lr_accuracy, rf_accuracy],
    'Лучшие параметры': [
        str(knn_grid.best_params_),
        str(svm_grid.best_params_),
        str(dt_grid.best_params_),
        str(lr_grid.best_params_),
        str(rf_grid.best_params_)
    ]
})

results = results.sort_values('Accuracy на тесте', ascending=False)

print("\n" + "="*100)
print("СРАВНЕНИЕ АЛГОРИТМОВ КЛАССИФИКАЦИИ")
print("="*100)
print(results.to_string(index=False))
print("="*100)

## 5. Сохранение лучшей модели

In [ ]:
# Находим лучшую модель
models = {
    'KNN': (knn_grid, knn_accuracy),
    'SVM': (svm_grid, svm_accuracy),
    'DT': (dt_grid, dt_accuracy),
    'LR': (lr_grid, lr_accuracy),
    'RF': (rf_grid, rf_accuracy)
}

best_model_name = max(models.items(), key=lambda x: x[1][1])[0]
best_model = models[best_model_name][0]
best_accuracy = models[best_model_name][1]

print(f"Лучшая модель: {best_model_name} с accuracy = {best_accuracy:.4f}")

# Сохранение модели
joblib.dump(best_model, 'best_classifier.joblib')
print("Модель сохранена в файл 'best_classifier.joblib'")

## 6. Проверка импорта модели

In [ ]:
# Загрузка модели
loaded_model = joblib.load('best_classifier.joblib')
print("Модель успешно загружена")

# Проверка на тестовых данных
loaded_predictions = loaded_model.predict(X_test[:5])
original_predictions = best_model.predict(X_test[:5])

print(f"\nПредсказания загруженной модели: {loaded_predictions}")
print(f"Предсказания оригинальной модели: {original_predictions}")
print(f"Результаты совпадают: {np.array_equal(loaded_predictions, original_predictions)}")

# Метрики на всей тестовой выборке
loaded_accuracy = accuracy_score(y_test, loaded_model.predict(X_test))
print(f"\nAccuracy загруженной модели: {loaded_accuracy:.4f}")
print(f"\nОтчет классификации:")
print(classification_report(y_test, loaded_model.predict(X_test), target_names=iris.target_names))

## 7. Выводы

In [ ]:
print("\n📊 ВЫВОДЫ:")
print("="*70)
print(f"1. Лучший алгоритм: {best_model_name} с accuracy = {best_accuracy:.4f}")
print(f"2. Все модели показали высокую точность на датасете Iris")
print(f"3. Модель успешно сохранена и загружена обратно")
print(f"4. Загруженная модель работает корректно и дает те же результаты")
print("="*70)